In [ ]:
!pip install TikTokApi
!playwright install
!playwright install-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.4/65.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 24.0 MB/s eta 0:00:00
(node:2163) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
167.3 MiB [] 0% 0.0s167.3 MiB [] 0% 41.4s167.3 MiB [] 0% 17.5s167.3 MiB [] 0% 14.5s167.3 MiB [] 0% 11.7s167.3 MiB [] 1% 5.8s167.3 MiB [] 2% 4.6s167.3 MiB [] 2% 3.8s167.3 MiB [] 3% 3.5s167.3 MiB [] 3% 3.6s167.3 MiB [] 4% 3.5s167.3 MiB [] 5% 3.1s167.3 MiB [] 6% 2.8s167.3 MiB [] 7% 2.6s167.3 MiB [] 8% 2.5s167.3 MiB [] 9% 2.4s167.3 MiB [] 10% 2.3s167.3 MiB [] 11% 2.2s167.3 MiB [] 12% 2.1s167.3 MiB [] 13% 2.0s167.3 MiB [] 14% 2.0s167.3 MiB [] 15% 2.0s167.3 MiB [] 16% 2.0s167.3 MiB [] 17% 2.0s167.3 MiB [] 17% 1.9s167.3 MiB [] 18% 1.9s167.3 MiB

In [ ]:
import asyncio
from TikTokApi import TikTokApi
import pandas as pd
from google.colab import files
import os

async def crawl_and_export_csv(keyword, video_count=10):
    hashtag_name = keyword.replace(" ", "").replace("#", "")
    print(f"Đang tìm kiếm với hashtag: #{hashtag_name}")

    api = TikTokApi()
    video_data_list = []

    try:
        # Khởi tạo session
        await api.create_sessions(
            num_sessions=1,
            sleep_after=10,
            browser='webkit',
            headless=True
        )

        tag = api.hashtag(name=hashtag_name)
        count = 0
        print("Đang lấy dữ liệu... (vui lòng chờ)")

        async for video in tag.videos(count=video_count):
            v_info = video.as_dict
            author = v_info.get("author", {}).get("uniqueId")
            v_id = v_info.get("id")
            stats = v_info.get("stats", {})

            # Thu thập các trường thông tin quan trọng
            record = {
                "Video ID": v_id,
                "URL": f"https://www.tiktok.com/@{author}/video/{v_id}",
                "Tác giả": author,
                "Mô tả": v_info.get("desc", ""),
                "Lượt tim": stats.get("diggCount", 0),
                "Lượt xem": stats.get("playCount", 0),
                "Lượt chia sẻ": stats.get("shareCount", 0),
                "Lượt bình luận": stats.get("commentCount", 0)
            }
            video_data_list.append(record)

            print(f"- Đã lấy video {count + 1}: {record['URL']}")

            count += 1
            if count >= video_count:
                break

        if video_data_list:
            # Tạo DataFrame và xuất CSV
            df = pd.DataFrame(video_data_list)
            csv_filename = f"tiktok_data_{hashtag_name}.csv"
            # utf-8-sig giúp Excel hiển thị đúng tiếng Việt
            df.to_csv(csv_filename, index=False, encoding='utf-8-sig')

            print(f"\n[XONG] Đã thu thập {len(video_data_list)} video.")
            print(f"Đang tự động tải file '{csv_filename}' xuống...")
            files.download(csv_filename)
        else:
            print("\n[THÔNG BÁO] Không có dữ liệu để xuất file (có thể do bị chặn IP).")

    except Exception as e:
        print(f"Lỗi: {e}")
    finally:
        await api.close_sessions()

# --- CẤU HÌNH CHẠY ---
keyword = "xe hơi" # @param {type:"string"}
video_limit = 20 # @param {type:"integer"}

await crawl_and_export_csv(keyword, video_count=video_limit)

Đang tìm kiếm với hashtag: #xehơi
Đang lấy dữ liệu... (vui lòng chờ)
- Đã lấy video 1: https://www.tiktok.com/@zoom6car/video/7613253519184842005
- Đã lấy video 2: https://www.tiktok.com/@anhchuxuongdeptrai/video/7612659344768896264
- Đã lấy video 3: https://www.tiktok.com/@vungocthanhh/video/7353136888313679124
- Đã lấy video 4: https://www.tiktok.com/@quenha.6/video/7608966011928317215
- Đã lấy video 5: https://www.tiktok.com/@cuongnuocmy/video/7427307108728687915
- Đã lấy video 6: https://www.tiktok.com/@cuongnuocmy/video/7377548592749317418
- Đã lấy video 7: https://www.tiktok.com/@autocarclub/video/7275932066582154503
- Đã lấy video 8: https://www.tiktok.com/@thaidacde1999/video/7544259066764479762
- Đã lấy video 9: https://www.tiktok.com/@sieuxe.360/video/7612687977428339975
- Đã lấy video 10: https://www.tiktok.com/@gatgu_channel/video/7456391235574517008
- Đã lấy video 11: https://www.tiktok.com/@chuyen_ve_oto/video/7605222950781898004
- Đã lấy video 12: https://www.tiktok.com/

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#Tách lấy cột URL
import pandas as pd
from google.colab import files
import io

# 1. Tải file CSV lên từ máy tính
print("Hãy chọn file CSV bạn vừa tải xuống:")
uploaded = files.upload()

# 2. Đọc file vừa tải lên
for filename in uploaded.keys():
    df_imported = pd.read_csv(io.BytesIO(uploaded[filename]))

    # 3. Kiểm tra và lấy cột URL
    if 'URL' in df_imported.columns:
        url_column = df_imported[['URL']]
        print(f"\n--- Đã trích xuất cột URL từ file: {filename} ---")
        display(url_column)

        # 4. Lưu thành file mới và tự động tải về
        output_filename = f"extracted_urls_{filename}"
        url_column.to_csv(output_filename, index=False, encoding='utf-8-sig')

        print(f"Đang tự động tải file kết quả '{output_filename}' xuống...")
        files.download(output_filename)
    else:
        print(f"\n[LỖI] Không tìm thấy cột 'URL' trong file {filename}.")
        print("Các cột hiện có:", df_imported.columns.tolist())

Hãy chọn file CSV bạn vừa tải xuống:


Saving tiktok_data_xehơi.csv to tiktok_data_xehơi (1).csv

--- Đã trích xuất cột URL từ file: tiktok_data_xehơi (1).csv ---


,URL
0,https://www.tiktok.com/@zoom6car/video/7613253...
1,https://www.tiktok.com/@anhchuxuongdeptrai/vid...
2,https://www.tiktok.com/@vungocthanhh/video/735...
3,https://www.tiktok.com/@quenha.6/video/7608966...
4,https://www.tiktok.com/@cuongnuocmy/video/7427...
5,https://www.tiktok.com/@cuongnuocmy/video/7377...
6,https://www.tiktok.com/@autocarclub/video/7275...
7,https://www.tiktok.com/@thaidacde1999/video/75...
8,https://www.tiktok.com/@sieuxe.360/video/76126...
9,https://www.tiktok.com/@gatgu_channel/video/74...


Đang tự động tải file kết quả 'extracted_urls_tiktok_data_xehơi (1).csv' xuống...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#Tổng hợp công đoạn crawl
import pandas as pd
import requests
import re
import time
import random
from google.colab import files

def run_full_comment_crawl(target_df, comments_limit=30):
    all_comments_data = []
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Referer": "https://www.tiktok.com/"
    }

    # 1. Trích xuất URL
    if 'URL' not in target_df.columns:
        print("Lỗi: Không tìm thấy cột 'URL' trong dữ liệu.")
        return

    urls = target_df['URL'].tolist()
    print(f"Bắt đầu quét {len(urls)} video...")

    # 2. Duyệt qua từng URL và lấy Comment
    for i, url in enumerate(urls):
        try:
            match = re.search(r'video/(\d+)', url)
            if not match: continue
            video_id = match.group(1)

            print(f"[{i+1}/{len(urls)}] Đang lấy comment video ID: {video_id}...")
            api_url = f"https://www.tiktok.com/api/comment/list/?aid=1988&aweme_id={video_id}&count={comments_limit}&cursor=0"

            response = requests.get(api_url, headers=headers, timeout=15)

            if response.status_code == 200:
                data = response.json()
                comments = data.get('comments', [])

                for c in comments:
                    all_comments_data.append({
                        "Video ID": video_id,
                        "Comment Content": c.get("text", ""),
                        "User": c.get("user", {}).get("unique_id", ""),
                        "Like Count": c.get("digg_count", 0)
                    })
                print(f"  -> Thành công: Lấy được {len(comments)} bình luận.")
            else:
                print(f"  -> Thất bại: Mã lỗi {response.status_code}")

            # Nghỉ giữa các lần gọi để tránh bị chặn
            time.sleep(random.uniform(1.5, 3.0))

        except Exception as e:
            print(f"  -> Lỗi khi xử lý URL {url}: {e}")
            continue

    # 3. Xuất file và tải về
    if all_comments_data:
        df_result = pd.DataFrame(all_comments_data)
        output_name = "tiktok_all_comments.csv"
        df_result.to_csv(output_name, index=False, encoding='utf-8-sig')
        print(f"\n--- HOÀN THÀNH ---")
        print(f"Tổng số bình luận thu thập: {len(df_result)}")
        print(f"Đang tải file '{output_name}'...")
        files.download(output_name)
    else:
        print("\nKhông thu thập được bình luận nào.")

# Kiểm tra nếu biến df_imported (từ bước upload) tồn tại
if 'df_imported' in globals():
    run_full_comment_crawl(df_imported, comments_limit=30)
else:
    print("Vui lòng chạy ô 'Tải file CSV lên' trước khi thực hiện bước này.")

Bắt đầu quét 20 video...
[1/20] Đang lấy comment video ID: 7613253519184842005...
  -> Thành công: Lấy được 1 bình luận.
[2/20] Đang lấy comment video ID: 7612659344768896264...
  -> Thành công: Lấy được 3 bình luận.
[3/20] Đang lấy comment video ID: 7353136888313679124...
  -> Thành công: Lấy được 29 bình luận.
[4/20] Đang lấy comment video ID: 7608966011928317215...
  -> Thành công: Lấy được 28 bình luận.
[5/20] Đang lấy comment video ID: 7427307108728687915...
  -> Thành công: Lấy được 29 bình luận.
[6/20] Đang lấy comment video ID: 7377548592749317418...
  -> Thành công: Lấy được 29 bình luận.
[7/20] Đang lấy comment video ID: 7275932066582154503...
  -> Thành công: Lấy được 29 bình luận.
[8/20] Đang lấy comment video ID: 7544259066764479762...
  -> Thành công: Lấy được 28 bình luận.
[9/20] Đang lấy comment video ID: 7612687977428339975...
  -> Thành công: Lấy được 1 bình luận.
[10/20] Đang lấy comment video ID: 7456391235574517008...
  -> Thành công: Lấy được 30 bình luận.
[11/20]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#**PHIÊN BẢN CẢI THIỆN**
- Chuẩn hóa keyword nhập đầu vào, chuyển thành hashtag

- Tìm kiếm dựa trên hagtash

- Lọc cột URL, tự động crawl comment dựa trên URL

- Chuẩn hóa comment loại bỏ icon, khoảng trống không cần thiết

In [10]:
#Cài đặt các thư viện cần thiết hỗ trợ crawl dữ liệu trên TikTok
!pip install TikTokApi
!pip install pandas

!playwright install
!playwright install-deps

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

KeyboardInterrupt: 

In [3]:
import asyncio
from TikTokApi import TikTokApi
import pandas as pd
from google.colab import files
import re
import unicodedata
from datetime import datetime


# ==============================
# 1. Chuẩn hóa keyword đầu vào
# ==============================
def normalize_keyword(keyword):

    keyword = keyword.strip().lower()

    # bỏ ký tự đặc biệt
    keyword = re.sub(r'[^\w\s]', '', keyword)

    # bỏ dấu tiếng Việt
    keyword = ''.join(
        c for c in unicodedata.normalize('NFD', keyword)
        if unicodedata.category(c) != 'Mn'
    )

    hashtag_name = keyword.replace(" ", "")

    return hashtag_name


# ==============================
# 2. Tạo session an toàn
# ==============================
async def create_session_safe(api):

    for attempt in range(3):
        try:
            await api.create_sessions(
                num_sessions=1,
                sleep_after=10,
                browser="webkit",
                headless=True
            )
            return
        except Exception as e:
            print(f"Lỗi tạo session (lần {attempt+1})")
            await asyncio.sleep(5)

    raise Exception("Không thể tạo session TikTok")


# ==============================
# 3. Crawl video theo hashtag
# ==============================
async def crawl_and_export_csv(keyword, video_count=10):

    hashtag_name = normalize_keyword(keyword)

    print(f"Đang tìm kiếm hashtag: #{hashtag_name}")

    api = TikTokApi()
    video_data_list = []

    try:

        await create_session_safe(api)

        tag = api.hashtag(name=hashtag_name)

        count = 0

        print("Đang thu thập dữ liệu...")

        async for video in tag.videos(count=video_count):

            try:

                v_info = video.as_dict

                author = v_info.get("author", {}).get("uniqueId")
                v_id = v_info.get("id")

                # bỏ video thiếu dữ liệu
                if not author or not v_id:
                    continue

                stats = v_info.get("stats", {})

                video_url = f"https://www.tiktok.com/@{author}/video/{v_id}"

                record = {

                    "Video ID": v_id,

                    "Video URL": video_url,

                    "Tác giả": author,

                    "Mô tả": v_info.get("desc", ""),

                    "Thời gian đăng": v_info.get("createTime"),

                    "Thời lượng": v_info.get("video", {}).get("duration", 0),

                    "Lượt tim": stats.get("diggCount", 0),

                    "Lượt xem": stats.get("playCount", 0),

                    "Lượt chia sẻ": stats.get("shareCount", 0),

                    "Lượt bình luận": stats.get("commentCount", 0),

                    "Hashtag": hashtag_name
                }

                video_data_list.append(record)

                print(f"[{count+1}/{video_count}] {video_url}")

                count += 1

                if count >= video_count:
                    break

                await asyncio.sleep(1)

            except Exception as e:
                print("Lỗi khi xử lý video:", e)
                continue

        if video_data_list:

            df = pd.DataFrame(video_data_list)

            # loại bỏ trùng
            df = df.drop_duplicates(subset="Video ID")

            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

            csv_filename = f"tiktok_data_{hashtag_name}_{timestamp}.csv"

            df.to_csv(
                csv_filename,
                index=False,
                encoding="utf-8-sig"
            )

            print("\n[XONG] Đã thu thập", len(df), "video")
            print("Đang tải file CSV...")

            files.download(csv_filename)

        else:
            print("Không có dữ liệu (có thể bị TikTok chặn).")

    except Exception as e:
        print("Lỗi:", e)

    finally:
        await api.close_sessions()


# ==============================
# 4. Cấu hình chạy
# ==============================

keyword = "du lịch %&^*^"  # @param {type:"string"}

video_limit = 30  # @param {type:"integer"}

await crawl_and_export_csv(keyword, video_count=video_limit)

Đang tìm kiếm hashtag: #dulich
Đang thu thập dữ liệu...
[1/30] https://www.tiktok.com/@thuykieu_2802/video/7589906430824795399
[2/30] https://www.tiktok.com/@thuykieu_2802/video/7535780207366704391
[3/30] https://www.tiktok.com/@phuongtrip11/video/7513501547175873800
[4/30] https://www.tiktok.com/@kaiph__/video/7365128711143902482
[5/30] https://www.tiktok.com/@sau5894/video/7558108154458819848
[6/30] https://www.tiktok.com/@zavatravel/video/7277304545565232392
[7/30] https://www.tiktok.com/@bunchaiue/video/7591781896086998292
[8/30] https://www.tiktok.com/@minhhang_buingoc/video/7276089385718762753
[9/30] https://www.tiktok.com/@kid_2509/video/7412277687949085959
[10/30] https://www.tiktok.com/@gowith_we/video/7410631152202943745
[11/30] https://www.tiktok.com/@zavatravel/video/7289118632284654856
[12/30] https://www.tiktok.com/@gowith_we/video/7491660880023670034
[13/30] https://www.tiktok.com/@trangg1423/video/7576105755024149767
[14/30] https://www.tiktok.com/@pupu.haha/video/76043

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
#Tách lấy cột Video URL
import pandas as pd
from google.colab import files
import io

# 1. Tải file CSV lên từ máy tính
print("Hãy chọn file CSV bạn vừa tải xuống:")
uploaded = files.upload()

# 2. Đọc file vừa tải lên
for filename in uploaded.keys():
    df_imported = pd.read_csv(io.BytesIO(uploaded[filename]))

    # 3. Kiểm tra và lấy cột URL
    if 'URL' in df_imported.columns:
        url_column = df_imported[['Video URL']]
        print(f"\n--- Đã trích xuất cột URL từ file: {filename} ---")
        display(url_column)

        # 4. Lưu thành file mới và tự động tải về
        output_filename = f"extracted_urls_{filename}"
        url_column.to_csv(output_filename, index=False, encoding='utf-8-sig')

        print(f"Đang tự động tải file kết quả '{output_filename}' xuống...")
        files.download(output_filename)
    else:
        print(f"\n[LỖI] Không tìm thấy cột 'URL' trong file {filename}.")
        print("Các cột hiện có:", df_imported.columns.tolist())

Hãy chọn file CSV bạn vừa tải xuống:


Saving tiktok_data_dulich_20260306_005750.csv to tiktok_data_dulich_20260306_005750 (2).csv

[LỖI] Không tìm thấy cột 'URL' trong file tiktok_data_dulich_20260306_005750 (2).csv.
Các cột hiện có: ['Video ID', 'Video URL', 'Tác giả', 'Mô tả', 'Thời gian đăng', 'Thời lượng', 'Lượt tim', 'Lượt xem', 'Lượt chia sẻ', 'Lượt bình luận', 'Hashtag']


In [6]:
import pandas as pd
from google.colab import files
import io

# 1. Upload file CSV
print("Hãy chọn file CSV do crawler TikTok tạo:")
uploaded = files.upload()

# 2. Đọc file CSV
for filename in uploaded.keys():

    df_imported = pd.read_csv(io.BytesIO(uploaded[filename]))

    # 3. Kiểm tra cột Video URL
    if 'Video URL' in df_imported.columns:

        url_column = df_imported[['Video URL']].copy()

        # chuẩn hóa URL
        url_column['Video URL'] = url_column['Video URL'].astype(str).str.strip()

        # loại bỏ dòng rỗng
        url_column = url_column[url_column['Video URL'] != ""]

        print(f"\n--- Đã trích xuất cột Video URL từ file: {filename} ---")
        display(url_column)

        # 4. Lưu file mới
        output_filename = f"extracted_video_urls_{filename}"

        url_column.to_csv(
            output_filename,
            index=False,
            encoding='utf-8-sig'
        )

        print(f"Đang tải file kết quả '{output_filename}' xuống...")

        files.download(output_filename)

    else:

        print(f"\n[LỖI] Không tìm thấy cột 'Video URL' trong file {filename}")
        print("Các cột hiện có:", df_imported.columns.tolist())

Hãy chọn file CSV do crawler TikTok tạo:


Saving tiktok_data_dulich_20260306_005750.csv to tiktok_data_dulich_20260306_005750 (3).csv

--- Đã trích xuất cột Video URL từ file: tiktok_data_dulich_20260306_005750 (3).csv ---


,Video URL
0,https://www.tiktok.com/@thuykieu_2802/video/75...
1,https://www.tiktok.com/@thuykieu_2802/video/75...
2,https://www.tiktok.com/@phuongtrip11/video/751...
3,https://www.tiktok.com/@kaiph__/video/73651287...
4,https://www.tiktok.com/@sau5894/video/75581081...
5,https://www.tiktok.com/@zavatravel/video/72773...
6,https://www.tiktok.com/@bunchaiue/video/759178...
7,https://www.tiktok.com/@minhhang_buingoc/video...
8,https://www.tiktok.com/@kid_2509/video/7412277...
9,https://www.tiktok.com/@gowith_we/video/741063...


Đang tải file kết quả 'extracted_video_urls_tiktok_data_dulich_20260306_005750 (3).csv' xuống...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
import pandas as pd
import requests
import re
import time
import random
from google.colab import files

def run_full_comment_crawl(target_df, comments_limit=30):

    all_comments_data = []

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://www.tiktok.com/"
    }

    # =============================
    # 1. Xác định cột URL
    # =============================

    if 'Video URL' in target_df.columns:
        url_column = 'Video URL'

    elif 'URL' in target_df.columns:
        url_column = 'URL'

    else:
        print("Lỗi: Không tìm thấy cột URL trong dữ liệu")
        print("Các cột hiện có:", target_df.columns.tolist())
        return


    urls = target_df[url_column].dropna().tolist()

    print(f"Bắt đầu quét {len(urls)} video...")


    # =============================
    # 2. Crawl comment từng video
    # =============================

    for i, url in enumerate(urls):

        try:

            url = str(url).strip()

            match = re.search(r'video/(\d+)', url)

            if not match:
                print("  -> Không lấy được video_id")
                continue

            video_id = match.group(1)

            print(f"[{i+1}/{len(urls)}] Video ID: {video_id}")

            api_url = f"https://www.tiktok.com/api/comment/list/?aid=1988&aweme_id={video_id}&count={comments_limit}&cursor=0"

            response = requests.get(api_url, headers=headers, timeout=15)

            if response.status_code == 200:

                data = response.json()

                comments = data.get("comments", [])

                print(f"  -> Lấy được {len(comments)} comment")

                for c in comments:

                    all_comments_data.append({

                        "Video ID": video_id,

                        "Video URL": url,

                        "Comment Content": c.get("text", ""),

                        "User": c.get("user", {}).get("unique_id", ""),

                        "Like Count": c.get("digg_count", 0)

                    })

            else:

                print(f"  -> Lỗi HTTP {response.status_code}")


            time.sleep(random.uniform(1.5, 3.5))


        except Exception as e:

            print(f"  -> Lỗi khi xử lý URL {url}: {e}")

            continue


    # =============================
    # 3. Xuất CSV
    # =============================

    if all_comments_data:

        df_result = pd.DataFrame(all_comments_data)

        output_name = "tiktok_all_comments.csv"

        df_result.to_csv(

            output_name,
            index=False,
            encoding='utf-8-sig'
        )

        print("\n--- HOÀN THÀNH ---")

        print("Tổng số comment:", len(df_result))

        files.download(output_name)

    else:

        print("\nKhông thu thập được comment nào.")



# =============================
# 4. Chạy crawler
# =============================

if 'df_imported' in globals():

    run_full_comment_crawl(df_imported, comments_limit=30)

else:

    print("Vui lòng chạy bước upload CSV trước.")

Bắt đầu quét 30 video...
[1/30] Video ID: 7589906430824795399
  -> Lấy được 30 comment
[2/30] Video ID: 7535780207366704391
  -> Lấy được 29 comment
[3/30] Video ID: 7513501547175873800
  -> Lấy được 30 comment
[4/30] Video ID: 7365128711143902482
  -> Lấy được 30 comment
[5/30] Video ID: 7558108154458819848
  -> Lấy được 20 comment
[6/30] Video ID: 7277304545565232392
  -> Lấy được 30 comment
[7/30] Video ID: 7591781896086998292
  -> Lấy được 30 comment
[8/30] Video ID: 7276089385718762753
  -> Lấy được 30 comment
[9/30] Video ID: 7412277687949085959
  -> Lấy được 30 comment
[10/30] Video ID: 7410631152202943745
  -> Lấy được 30 comment
[11/30] Video ID: 7289118632284654856
  -> Lấy được 29 comment
[12/30] Video ID: 7491660880023670034
  -> Lấy được 30 comment
[13/30] Video ID: 7576105755024149767
  -> Lấy được 30 comment
[14/30] Video ID: 7604385762389413140
  -> Lấy được 30 comment
[15/30] Video ID: 7280720171113712897
  -> Lấy được 30 comment
[16/30] Video ID: 7162559333165731098
 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
import pandas as pd
import re
from google.colab import files
import io

print("Hãy chọn file CSV comment:")
uploaded = files.upload()

# =============================
# Hàm xoá được nhiều emoji trong comment hơn
# =============================

def remove_emoji(text):

    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F700-\U0001F77F"
        "\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FAFF"
        "\U00002600-\U000026FF"
        "\U00002700-\U000027BF"
        "]+",
        flags=re.UNICODE
    )

    return emoji_pattern.sub("", text)


# =============================
# Chuẩn hoá comment
# =============================

def normalize_comment(text):

    if pd.isna(text):
        return ""

    text = str(text)

    # xoá emoji
    text = remove_emoji(text)

    # xoá xuống dòng
    text = text.replace("\n", " ")

    # xoá khoảng trắng dư
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# =============================
# Xử lý file
# =============================

for filename in uploaded.keys():

    df = pd.read_csv(io.BytesIO(uploaded[filename]))

    if "Comment Content" not in df.columns:

        print("Không tìm thấy cột 'Comment Content'")
        print("Các cột hiện có:", df.columns.tolist())

    else:

        print("Bắt đầu chuẩn hoá comment...")

        df["Comment Clean"] = df["Comment Content"].apply(normalize_comment)

        # xoá comment rỗng
        df = df[df["Comment Clean"] != ""]

        print("Số comment sau khi làm sạch:", len(df))

        output_file = "tiktok_comments_cleaned.csv"

        df.to_csv(
            output_file,
            index=False,
            encoding="utf-8-sig"
        )

        print("Đã hoàn thành chuẩn hoá!")

        files.download(output_file)

Hãy chọn file CSV comment:


Saving tiktok_all_comments (3).csv to tiktok_all_comments (3) (1).csv
Bắt đầu chuẩn hoá comment...
Số comment sau khi làm sạch: 810
Đã hoàn thành chuẩn hoá!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>